# NLP – Unit 2: Representación por Frecuencias

Ejercicios sobre BoW y TF-IDF

In [ ]:
# === Setup y utilidades ===
from collections import Counter
import math
import re
import pandas as pd

In [ ]:
# Dataset de juguete
docs = [
    "El gato negro se sentó en el sofá.",
    "El perro blanco jugaba en el jardín.",
    "El gato y el perro son amigos."
]

In [ ]:
def normalize_token(token):
    # Sólo lowercase
    return token.lower()

def simple_tokenize(text):
    # Tokenizador sencillo: separa por secuencias de letras (incluye acentos)
    tokens = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", text)
    tokens = [normalize_token(t) for t in tokens if t.strip()]
    return tokens

In [ ]:
# Vista previa
for i, d in enumerate(docs, 1):
    print(f"Doc {i}: {d}")
    print("Tokens:", simple_tokenize(d))
    print("-"*60)

Doc 1: El gato negro se sentó en el sofá.
Tokens: ['el', 'gato', 'negro', 'se', 'sentó', 'en', 'el', 'sofá']
------------------------------------------------------------
Doc 2: El perro blanco jugaba en el jardín.
Tokens: ['el', 'perro', 'blanco', 'jugaba', 'en', 'el', 'jardín']
------------------------------------------------------------
Doc 3: El gato y el perro son amigos.
Tokens: ['el', 'gato', 'y', 'el', 'perro', 'son', 'amigos']
------------------------------------------------------------


```python
tokens = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", text)
```
Usa **expresiones regulares (regex)** para hacer una **tokenización muy simple** del texto. Vamos por partes:

* `re.findall(...)` → busca **todas** las coincidencias en la cadena `text` y devuelve una lista.
* El patrón `r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+"` significa:

  * `[ ... ]` → conjunto de caracteres permitidos.
  * `A-Z` → letras mayúsculas en inglés.
  * `a-z` → letras minúsculas en inglés.
  * `ÁÉÍÓÚÜÑ` → incluye explícitamente letras mayúsculas con tildes y la Ñ.
  * `áéíóúüñ` → lo mismo para minúsculas.
  * `+` → una o más repeticiones consecutivas de esos caracteres (es decir, forma “palabras”).

En resumen, extrae todas las secuencias de letras del texto (incluyendo tildes y ñ), ignorando números, signos de puntuación y espacios.

## Ejercicio 1 – Construcción de la matriz Bag of Words (BoW)

**Objetivo:** representar documentos mediante conteos de palabras.

**Tareas:**
1. Construye el **vocabulario** (lista ordenada de palabras únicas) a partir de `docs`.
2. Representa cada documento como un vector BoW de tamaño `len(vocab)`.
3. Construye la **matriz BoW** (lista de listas o matriz `numpy`) con los documentos como filas.
4. Verifica con asserts sencillos.

**Pistas:**
- Usa `simple_tokenize`.
- Ordena el vocabulario alfabéticamente para resultados reproducibles.

In [ ]:
def bow_vector(tokens, tok2idx):
    counts = Counter(tokens)
    return [counts.get(t,0) for t in tok2idx]

In [ ]:
docs_tokens = [t for d in docs for t in simple_tokenize(d)]
vocab = sorted(set(docs_tokens))
tok2idx = {t:i for i,t in enumerate(vocab)}

print("doc_tokens = ", docs_tokens, "\n\ntok2idx= ", tok2idx)

doc_tokens =  ['el', 'gato', 'negro', 'se', 'sentó', 'en', 'el', 'sofá', 'el', 'perro', 'blanco', 'jugaba', 'en', 'el', 'jardín', 'el', 'gato', 'y', 'el', 'perro', 'son', 'amigos'] 

tok2idx=  {'amigos': 0, 'blanco': 1, 'el': 2, 'en': 3, 'gato': 4, 'jardín': 5, 'jugaba': 6, 'negro': 7, 'perro': 8, 'se': 9, 'sentó': 10, 'sofá': 11, 'son': 12, 'y': 13}


In [ ]:
# Matriz Bags of Words
docs_tokens = [simple_tokenize(d) for d in docs]
bow_mat = [bow_vector(tokens, tok2idx) for tokens in docs_tokens]

print("docs_tokens = ", docs_tokens)
print("\nVocabulario:", vocab)
print("\nMatriz BoW:")
for row in bow_mat:
    print(row)

docs_tokens =  [['el', 'gato', 'negro', 'se', 'sentó', 'en', 'el', 'sofá'], ['el', 'perro', 'blanco', 'jugaba', 'en', 'el', 'jardín'], ['el', 'gato', 'y', 'el', 'perro', 'son', 'amigos']]

Vocabulario: ['amigos', 'blanco', 'el', 'en', 'gato', 'jardín', 'jugaba', 'negro', 'perro', 'se', 'sentó', 'sofá', 'son', 'y']

Matriz BoW:
[0, 0, 2, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0]
[0, 1, 2, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0]
[1, 0, 2, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1]


In [ ]:
bow_mat

[[0, 0, 2, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0],
 [0, 1, 2, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0],
 [1, 0, 2, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1]]

## Ejercicio 2 – Construcción de la matriz TF‑IDF

**Objetivo:** aplicar TF, IDF y TF‑IDF sobre `docs` y comparar con BoW.

**Definiciones:**
- $TF(t, d) = f_{t,d} /$ `(nº términos en d)`
- $IDF(t) = log(N / (1 + n_t))$ con `N = # documentos` y $n_t=$ `# docs que contienen t`
- $TF‑IDF(t, d) = TF(t, d) * IDF(t)$

Aparece el “+1” para evitar división por cero:
   * Si una palabra **no aparece en ningún documento** ($n_t = 0$), el denominador sería cero


### Variantes en la práctica

No hay una única forma de calcular IDF, y distintos paquetes usan pequeñas variaciones:

* **scikit-learn** usa:

  $$
  IDF(t) = \log\left(\frac{1 + N}{1 + n_t}\right) + 1
  $$

  (añade +1 en numerador y un desplazamiento +1 al resultado).

* **Versión clásica (sin suavizado):**

  $$
  IDF(t) = \log\left(\frac{N}{n_t}\right)
  $$

  usada en IR (Information Retrieval) tradicional.

**Tareas:**
1. Implementa funciones `tf`, `idf` y `tfidf`.
2. Construye la **matriz TF‑IDF** con el mismo orden de vocabulario del Ejercicio 1.
3. Valida propiedades básicas (p.ej., IDF de “el” ≈ bajo; IDF de “amigos” ≈ alto).


In [ ]:
def tf(term, tokens):
    return tokens.count(term) / len(tokens)

def idf(term, docs_tokens):
    N = len(docs_tokens)
    nt = sum(1 for d in docs_tokens if term in d)
    return math.log(N / (nt))

def tfidf(term, tokens, docs_tokens):
    return tf(term, tokens) * idf(term, docs_tokens)

In [ ]:
# Recordemos
for i, d in enumerate(docs, 1):
    print(f"Doc {i}: {d}")
    print("Tokens:", simple_tokenize(d))
    print("-"*60)

Doc 1: El gato negro se sentó en el sofá.
Tokens: ['el', 'gato', 'negro', 'se', 'sentó', 'en', 'el', 'sofá']
------------------------------------------------------------
Doc 2: El perro blanco jugaba en el jardín.
Tokens: ['el', 'perro', 'blanco', 'jugaba', 'en', 'el', 'jardín']
------------------------------------------------------------
Doc 3: El gato y el perro son amigos.
Tokens: ['el', 'gato', 'y', 'el', 'perro', 'son', 'amigos']
------------------------------------------------------------


In [ ]:
# Calcular matrices
tf_mat = [[tf(term, tokens) for term in vocab] for tokens in docs_tokens]
idf_vec = [idf(term, docs_tokens) for term in vocab]
tfidf_mat = [[tfidf(term, tokens, docs_tokens) for term in vocab] for tokens in docs_tokens]

df_tf = pd.DataFrame(tf_mat, columns=vocab)
df_idf = pd.DataFrame([idf_vec], columns=vocab)
df_tfidf = pd.DataFrame(tfidf_mat, columns=vocab)

display("tf = ", df_tf.round(3), "idf = ", df_idf.round(3), "tf-idf = ", df_tfidf.round(3) )

'tf = '

,amigos,blanco,el,en,gato,jardín,jugaba,negro,perro,se,sentó,sofá,son,y
0,0.000,0.000,0.250,0.125,0.125,0.000,0.000,0.125,0.000,0.125,0.125,0.125,0.000,0.000
1,0.000,0.143,0.286,0.143,0.000,0.143,0.143,0.000,0.143,0.000,0.000,0.000,0.000,0.000
2,0.143,0.000,0.286,0.000,0.143,0.000,0.000,0.000,0.143,0.000,0.000,0.000,0.143,0.143


'idf = '

,amigos,blanco,el,en,gato,jardín,jugaba,negro,perro,se,sentó,sofá,son,y
0,1.099,1.099,0.0,0.405,0.405,1.099,1.099,1.099,0.405,1.099,1.099,1.099,1.099,1.099


'tf-idf = '

,amigos,blanco,el,en,gato,jardín,jugaba,negro,perro,se,sentó,sofá,son,y
0,0.000,0.000,0.0,0.051,0.051,0.000,0.000,0.137,0.000,0.137,0.137,0.137,0.000,0.000
1,0.000,0.157,0.0,0.058,0.000,0.157,0.157,0.000,0.058,0.000,0.000,0.000,0.000,0.000
2,0.157,0.000,0.0,0.000,0.058,0.000,0.000,0.000,0.058,0.000,0.000,0.000,0.157,0.157


## Ejercicio 3 – Comparación BoW vs. TF‑IDF
**Objetivo:** observar la mejora práctica de TF‑IDF.

**Tareas:**
1. A partir de `bow_mat` y `tfidf_mat`, identifica:
   - Palabras con pesos similares en ambos modelos.
   - Palabras cuyo peso **cambia drásticamente** (p. ej., `'el'`)
2. Explica qué representación usarías para:
   - *Clasificación de documentos* (topic, spam/no spam)
   - *Búsqueda de información* (quién es más relevante para una consulta)

In [ ]:
def get_term_bow_column(term, vocab, bow_mat):
    j = vocab.index(term)
    return [row[j] for row in bow_mat]

def get_term_tfidf_column(term, vocab, tfidf_mat):
    j = vocab.index(term)
    return [row[j] for row in tfidf_mat]

In [ ]:
for term in vocab:
    bow_col = get_term_bow_column(term, vocab, bow_mat)
    tfidf_col = get_term_tfidf_column(term, vocab, tfidf_mat)
    print(f"\nTérmino: {term}")
    print(" BoW:   ", bow_col)
    print(" TF-IDF:", [round(x,4) for x in tfidf_col])


Término: amigos
 BoW:    [0, 0, 1]
 TF-IDF: [0.0, 0.0, 0.1569]

Término: blanco
 BoW:    [0, 1, 0]
 TF-IDF: [0.0, 0.1569, 0.0]

Término: el
 BoW:    [2, 2, 2]
 TF-IDF: [0.0, 0.0, 0.0]

Término: en
 BoW:    [1, 1, 0]
 TF-IDF: [0.0507, 0.0579, 0.0]

Término: gato
 BoW:    [1, 0, 1]
 TF-IDF: [0.0507, 0.0, 0.0579]

Término: jardín
 BoW:    [0, 1, 0]
 TF-IDF: [0.0, 0.1569, 0.0]

Término: jugaba
 BoW:    [0, 1, 0]
 TF-IDF: [0.0, 0.1569, 0.0]

Término: negro
 BoW:    [1, 0, 0]
 TF-IDF: [0.1373, 0.0, 0.0]

Término: perro
 BoW:    [0, 1, 1]
 TF-IDF: [0.0, 0.0579, 0.0579]

Término: se
 BoW:    [1, 0, 0]
 TF-IDF: [0.1373, 0.0, 0.0]

Término: sentó
 BoW:    [1, 0, 0]
 TF-IDF: [0.1373, 0.0, 0.0]

Término: sofá
 BoW:    [1, 0, 0]
 TF-IDF: [0.1373, 0.0, 0.0]

Término: son
 BoW:    [0, 0, 1]
 TF-IDF: [0.0, 0.0, 0.1569]

Término: y
 BoW:    [0, 0, 1]
 TF-IDF: [0.0, 0.0, 0.1569]
